# Resultados dos experimentos

Gera as tabelas e as figuras (PNG, PDF ou SVG) de um run em `runs/`. Usa o mesmo
código de `pixi run report`: os arquivos são idênticos aos da linha de comando e
ficam na pasta do run (`report_dev/`, `report_eval/` ou `report/`).

* **Kernel:** o Python do ambiente Pixi. No VS Code, *Select Kernel → Python
  Environments → `.pixi\envs\default`*; no navegador, `pixi run notebook`.
* **Relatório não chama a API** e não tem custo.
* A execução de etapas (seção 2) é opcional e fica desligada por padrão.

Sem editar nada, *Run All* mostra o relatório do último run usado. Para outro
run, defina `RUN` na célula de parâmetros.

In [ ]:
# Parâmetros — todos opcionais: com os valores None, o relatório é do último
# run usado ("*" em 'pixi run status'), na etapa mais avançada com respostas.
RUN = None                                # pasta em runs/, ex.: "completo_v1"; None = último run usado
ETAPA = None                              # multimodelo: "dev" ou "eval"; None = a mais avançada
EXPERIMENTO = None                        # só para criar um run novo (seção 2); senão vem do run
FORMATOS = ["png", "pdf"]                 # aceita "png", "pdf" e "svg"
RECORTE = []                              # ex.: ["select.temperatures=[0.0]", "select.models=[qwen38_27b]"]

# Etapas a executar ANTES do relatório, em ordem. Podem gerar CUSTOS na DeepInfra
# quando há requisições fora do cache, e exigem DEEPINFRA_TOKEN no ambiente.
# Exigem RUN; um run novo exige também EXPERIMENTO. Ex.: ["prepare", "dev"]; [] = só relatório.
EXECUTAR_ETAPAS = []
OVERRIDES_CRIACAO = []                    # só para run novo, ex.: ["experiment.models=[qwen38_27b]"]

## 1. Experimentos e runs disponíveis

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from fakebr import cli, project

raiz = project.project_root()
paths = project.Paths.from_config(
    raiz, project.compose(raiz, None, [o for o in RECORTE if o.startswith("paths.")])["paths"])


def relativo(caminho: Path) -> str:
    try:
        return Path(caminho).relative_to(raiz).as_posix()
    except ValueError:
        return str(caminho)


print("Projeto:", raiz)
cli.run_cli(["status", *[o for o in RECORTE if o.startswith("paths.")]])

## 2. Executar etapas (opcional)

⚠️ Só roda se `EXECUTAR_ETAPAS` não estiver vazio. Respostas já presentes no run
ou no cache são reaproveitadas sem custo; as demais são chamadas pagas.

In [ ]:
if EXECUTAR_ETAPAS and not RUN:
    raise ValueError("Defina RUN para executar etapas (elas podem gerar custos).")
for etapa in EXECUTAR_ETAPAS:
    novo = not (paths.runs / RUN).exists()
    print(f"\n>>> {RUN}: etapa {etapa}" + (" (run novo)" if novo else ""))
    cli.execute("experiment", EXPERIMENTO, RUN, etapa,
                [*RECORTE, *(OVERRIDES_CRIACAO if novo else [])])
if not EXECUTAR_ETAPAS:
    print("Nenhuma etapa executada (EXECUTAR_ETAPAS = []).")

## 3. Relatório

In [ ]:
resultado = cli.execute("report", EXPERIMENTO, RUN, ETAPA,
                        [*RECORTE, f"report.formats=[{','.join(FORMATOS)}]"])
print("Pasta do relatório:", relativo(resultado["report_dir"]))

## 4. Tabelas

In [ ]:
pd.set_option("display.max_columns", None)
for tabela in resultado["tables"]:
    if tabela.exists():
        display(Markdown(f"**{relativo(tabela)}**"))
        display(pd.read_csv(tabela))

## 5. Figuras

As imagens abaixo são os PNGs gerados; a lista final traz todos os formatos
pedidos em `FORMATOS`.

In [ ]:
for figura in resultado["figures"]:
    if figura.suffix == ".png":
        display(Markdown(f"**{figura.stem}**"))
        display(Image(filename=str(figura)))

print("Arquivos gerados:")
for figura in resultado["figures"]:
    print(" ", relativo(figura))

## 6. Custos acumulados (somente leitura)

In [ ]:
cli.run_cli(["costs", "--no-sync", *[o for o in RECORTE if o.startswith("paths.")]])